# 🌳 Python Binary Search Tree — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> A BST is a sorted filing cabinet built as a tree. Every drawer has a label.
> Anything smaller goes in the left drawer, anything larger goes in the right.
> To find a value, you never open every drawer — you follow the labels down
> and cut half the remaining search space at every step.

---

## 📋 Table of Contents

| # | Section |
|---|----------|
| 1 | [What Is a BST? The Visual Model](#1) |
| 2 | [Creating / Setup](#2) |
| 3 | [The Core API — All Operations](#3) |
| 4 | [Decision Map — When To Use What](#4) |
| 5 | [Pattern 1: Validate BST (LC 98)](#5) |
| 6 | [Pattern 2: Kth Smallest Element (LC 230)](#6) |
| 7 | [Pattern 3: Lowest Common Ancestor (LC 235)](#7) |
| 8 | [Pattern 4: Delete Node in BST (LC 450)](#8) |
| 9 | [The BST Decision Map](#9) |
| 10 | [Interview Cheat Sheet](#10) |

<a id='1'></a>
## 1. What Is a BST? The Visual Model

```
               BST — THE SORTED FILING CABINET

                        8          ← root (the top drawer label)
                      /   \
                    3       10     ← left < 8 < right (always)
                   / \       \
                  1   6       14   ← every subtree is also a valid BST
                     / \
                    4   7

  IN-ORDER TRAVERSAL (left → node → right) always gives SORTED output:
  1, 3, 4, 6, 7, 8, 10, 14

  BST PROPERTY (must hold for every node):
    ALL nodes in LEFT subtree  < current node
    ALL nodes in RIGHT subtree > current node
    (not just direct children — the ENTIRE subtree)

  SEARCH — like binary search on a sorted array:
    find(6): start at 8 → 6<8, go left → 6>3, go right → found!
    3 comparisons, not 7. O(log n) on a balanced tree.

  COMMON MISTAKES:
    ❌ checking only parent-child, not the full subtree range
    ❌ forgetting in-order traversal = sorted output
    ❌ using wrong bound direction on recursive calls
```

<a id='2'></a>
## 2. Creating / Setup

In [ ]:
# TreeNode definition — standard LeetCode node
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right
    def __repr__(self):
        return f"TreeNode({self.val})"

# make_tree(vals) — builds a tree from level-order list (BFS style)
# None in the list means no node at that position
from collections import deque

def make_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue = deque([root])
    i = 1
    while queue and i < len(vals):
        node = queue.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i])
            queue.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i])
            queue.append(node.right)
        i += 1
    return root

# tree_to_list(root) — serialize back to level-order for easy comparison
def tree_to_list(root):
    if not root:
        return []
    result, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node:
            result.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
        else:
            result.append(None)
    # strip trailing Nones for clean output
    while result and result[-1] is None:
        result.pop()
    return result

# inorder(root) — in-order traversal returns sorted list for a valid BST
def inorder(root):
    if not root:
        return []
    return inorder(root.left) + [root.val] + inorder(root.right)

# Build example BST: [8, 3, 10, 1, 6, None, 14, None, None, 4, 7]
bst = make_tree([8, 3, 10, 1, 6, None, 14, None, None, 4, 7])
print("in-order (should be sorted):", inorder(bst))  # [1, 3, 4, 6, 7, 8, 10, 14]
print("tree helpers loaded")

<a id='3'></a>
## 3. The Core API — All Operations

```
BST OPERATIONS          COMPLEXITY        WHAT IT DOES
────────────────────────────────────────────────────────────
search(root, val)        O(h)             follow left/right by value
insert(root, val)        O(h)             find leaf position, attach
delete(root, val)        O(h)             find, then 3-case handling
inorder(root)            O(n)             returns sorted array
validate(root)           O(n)             check BST property everywhere
lca(root, p, q)          O(h)             use BST property to navigate
kth_smallest(root, k)    O(h + k)         inorder stream, stop at k

h = tree height = O(log n) balanced, O(n) worst case (skewed tree)

THINGS YOU DO NOT DO:
❌  Validate BST by checking only node vs direct children
     (a node far down might violate a grandparent bound)
❌  Use in-order to validate — duplicates or subtle violations can pass
❌  Forget the 3 delete cases: leaf / one child / two children (use inorder successor)
❌  Mutate node values to "delete" — only restructure pointers
```

In [ ]:
# BST SEARCH — navigate left/right by value comparison
def bst_search(root, target):
    if not root:
        return None
    if target == root.val:
        return root
    if target < root.val:
        return bst_search(root.left, target)   # target is in the left filing drawer
    return bst_search(root.right, target)       # target is in the right filing drawer

# BST INSERT — find the leaf slot where the value belongs
def bst_insert(root, val):
    if not root:
        return TreeNode(val)                    # empty slot — place new node here
    if val < root.val:
        root.left = bst_insert(root.left, val) # belongs in the left subtree
    else:
        root.right = bst_insert(root.right, val) # belongs in the right subtree
    return root                                 # return root to rebuild the chain

# Demo: build BST by inserting one at a time
demo = None
for v in [5, 3, 7, 1, 4, 6, 8]:
    demo = bst_insert(demo, v)
print("inserted [5,3,7,1,4,6,8] in order:", inorder(demo))  # [1,3,4,5,6,7,8]

# Demo: search
found = bst_search(demo, 4)
print("search(4):", found)   # TreeNode(4)
missing = bst_search(demo, 9)
print("search(9):", missing) # None
print("BST operations demonstrated.")

<a id='4'></a>
## 4. Decision Map — When To Use What

```
SIGNAL IN THE PROBLEM                   WHAT TO DO
─────────────────────────────────────────────────────────────────
"validate BST"                          pass min/max bounds down recursion
"kth smallest in BST"                   in-order traversal, count to k
"lowest common ancestor in BST"         use BST property: navigate toward both
"delete node in BST"                    3-case: leaf / 1-child / 2-children
"insert into BST"                       recurse left or right, return root
"search in BST"                         O(h) — follow value comparison
"convert BST to sorted array"           in-order traversal
"range sum in BST"                      prune branches outside [low, high]
"closest value in BST"                  track best while navigating
"BST iterator"                          lazy in-order with explicit stack
```

<a id='5'></a>
## 5. 🧩 Pattern 1: Validate BST — LC 98

---

```
PROBLEM:
  Given the root of a binary tree, determine if it is a valid BST.

TRICK:
  Each node must satisfy a range [lo, hi] inherited from its ancestors.
  When you go left, the current node becomes the new upper bound (hi).
  When you go right, the current node becomes the new lower bound (lo).
  Check: lo < node.val < hi at every node.

SLOW MOTION TRACE on [5, 1, 4, None, None, 3, 6]:

  Tree:
       5
      / \
     1   4
        / \
       3   6

  validate(5,  lo=-inf, hi=+inf) → 5 in range ✓
    validate(1, lo=-inf, hi=5)   → 1 in range ✓
    validate(4, lo=5,   hi=+inf) → 4 > 5? NO → return False ✗

  Node 4 is in the right subtree of 5, but 4 < 5 — INVALID BST.

KEY INSIGHT:
  Every node inherits bounds from ALL its ancestors, not just its parent.
  The range-passing pattern is the only correct way to validate.

TIME:  O(n) — visit every node once
SPACE: O(h) — recursion stack, h = height
```

In [ ]:
def is_valid_bst(root):
    """
    LC 98 — Validate Binary Search Tree
    Approach: Pass inherited [lo, hi] bounds down every recursive call.
    Args:
        root (TreeNode): root of binary tree.
    Returns:
        bool: True if valid BST, False otherwise.
    Time:  O(n) — every node visited exactly once
    Space: O(h) — recursion depth = tree height
    """
    def validate(node, lo, hi):
        if not node:
            return True          # empty subtree is always valid
        if not (lo < node.val < hi):
            return False         # node.val violates inherited range → invalid
        # going left: current node becomes new upper bound
        # going right: current node becomes new lower bound
        return validate(node.left, lo, node.val) and validate(node.right, node.val, hi)

    return validate(root, float('-inf'), float('inf'))

# Slow motion on [2, 1, 3] (valid BST):
# validate(2, -inf, +inf) → 2 in range
#   validate(1, -inf, 2)  → 1 in range ✓
#   validate(3, 2, +inf)  → 3 in range ✓
# → True

# Slow motion on [5, 1, 4, None, None, 3, 6] (invalid):
# validate(5, -inf, +inf) → ok
#   validate(4, 5, +inf)  → 4 < 5 → False → short-circuit
# → False

def test_harness(fn):
    tests = [
        ([2, 1, 3], True),              # classic valid BST
        ([5, 1, 4, None, None, 3, 6], False),  # 4 in right subtree of 5 but 4<5
        ([5, 4, 6, None, None, 3, 7], False),  # 3 in right subtree of 5 but 3<5
        ([1], True),                    # single node
        ([10, 5, 15, None, None, 6, 20], False),  # 6 in right subtree of 10 but 6<10
        ([3, 1, 5, None, 2, None, None], True),   # valid
    ]
    passed = 0
    for *inputs, expected in tests:
        vals = inputs[0]
        root = make_tree(vals)
        got = fn(root)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | tree={vals} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(is_valid_bst)
print("is_valid_bst defined.")

<a id='6'></a>
## 6. 🧩 Pattern 2: Kth Smallest Element in BST — LC 230

---

```
PROBLEM:
  Given the root of a BST and an integer k, return the kth smallest value.

TRICK:
  In-order traversal of a BST yields values in ascending sorted order.
  Maintain a countdown counter. When counter reaches 0, record the answer.
  Use an iterative stack approach to stop early — no need to traverse everything.

SLOW MOTION TRACE on BST [3, 1, 4, None, 2], k=2:

  Tree:
       3
      / \
     1   4
      \
       2

  In-order sequence: 1, 2, 3, 4
  k=2 → answer is 2

  Iterative stack walkthrough:
  step  action               stack         k     result
    1   push left spine      [3, 1]        2     -
    2   pop 1, visit         [3]           1     -     (k went 2→1)
    3   push right of 1 (2)  [3, 2]        1     -
    4   push left spine(2)   [3, 2]        1     -     (2 has no left)
    5   pop 2, visit         [3]           0     2 ← k reached 0 → done!

KEY INSIGHT:
  In-order on BST = sorted. Stop at kth pop. Iterative stack enables early exit.

TIME:  O(h + k) — h to reach leftmost, k more pops
SPACE: O(h)     — stack depth = height
```

In [ ]:
def kth_smallest(root, k):
    """
    LC 230 — Kth Smallest Element in a BST
    Approach: Iterative in-order traversal; stop at kth node popped.
    Args:
        root (TreeNode): root of a valid BST.
        k (int): 1-indexed rank of the target smallest value.
    Returns:
        int: the kth smallest value in the BST.
    Time:  O(h + k) — h steps to reach leftmost leaf, k pops
    Space: O(h)     — explicit stack holds at most one root-to-leaf path
    """
    stack = []
    node = root
    while stack or node:
        # drill all the way left — smallest values live at the bottom-left
        while node:
            stack.append(node)
            node = node.left
        node = stack.pop()  # leftmost unvisited node = next smallest
        k -= 1              # count down toward the kth position
        if k == 0:
            return node.val # we've visited exactly k nodes → this is it
        node = node.right   # move to right subtree before going left again
    return -1               # k > number of nodes (shouldn't happen per constraints)

# Slow motion on [3, 1, 4, None, 2], k=2:
# node=3 → push 3, go left
# node=1 → push 1, go left
# node=None → stop drilling
# pop 1 → k=2-1=1, node=1.right=2
# node=2 → push 2, go left
# node=None → stop drilling
# pop 2 → k=1-1=0 → return 2

def test_harness(fn):
    tests = [
        ([3, 1, 4, None, 2], 1, 1),   # k=1 → smallest is 1
        ([3, 1, 4, None, 2], 2, 2),   # k=2 → second smallest is 2
        ([5, 3, 6, 2, 4, None, None, 1], 3, 3),  # k=3
        ([1], 1, 1),                  # single node
        ([4, 2, 6, 1, 3], 4, 4),      # sorted: 1,2,3,4,6 → k=4 is 4
    ]
    passed = 0
    for *inputs, expected in tests:
        vals, k = inputs
        root = make_tree(vals)
        got = fn(root, k)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | tree={vals} k={k} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(kth_smallest)
print("kth_smallest defined.")

<a id='7'></a>
## 7. 🧩 Pattern 3: Lowest Common Ancestor of BST — LC 235

---

```
PROBLEM:
  Given a BST and two nodes p and q, find their lowest common ancestor (LCA).
  LCA is the deepest node that has both p and q as descendants.

TRICK:
  Exploit the BST property:
  - If BOTH p and q are less than current node → LCA is in the LEFT subtree.
  - If BOTH p and q are greater than current node → LCA is in the RIGHT subtree.
  - Otherwise (p <= node <= q, or node == p, or node == q) → current node IS the LCA.
  This is O(h), not O(n) — no need to explore both sides.

SLOW MOTION TRACE on BST [6,2,8,0,4,7,9,None,None,3,5], p=2, q=8:

  node=6: p=2 < 6 and q=8 > 6 → split here → LCA = 6

SLOW MOTION TRACE same BST, p=2, q=4:

  node=6: both 2 and 4 < 6 → go LEFT
  node=2: both 2 and 4 >= 2 → p==node → LCA = 2

KEY INSIGHT:
  BST structure tells you which direction to walk. The first split point is the LCA.
  LCA in a generic binary tree requires O(n). BST makes it O(h).

TIME:  O(h) — walk at most one root-to-leaf path
SPACE: O(1) if iterative, O(h) if recursive
```

In [ ]:
def lowest_common_ancestor(root, p, q):
    """
    LC 235 — Lowest Common Ancestor of a Binary Search Tree
    Approach: Navigate using BST property — first split point is the LCA.
    Args:
        root (TreeNode): root of a valid BST.
        p (TreeNode): first target node.
        q (TreeNode): second target node.
    Returns:
        TreeNode: the lowest common ancestor node.
    Time:  O(h) — traverse at most one root-to-leaf path
    Space: O(1) — iterative, no call stack
    """
    node = root
    while node:
        if p.val < node.val and q.val < node.val:
            node = node.left   # both targets are in the left filing drawer
        elif p.val > node.val and q.val > node.val:
            node = node.right  # both targets are in the right filing drawer
        else:
            return node        # split point — current node is the ancestor of both
    return None

# Slow motion on [6,2,8,0,4,7,9], p=2, q=8:
# node=6: 2<6 and 8>6 → split → return node(6)

# Slow motion on [6,2,8,0,4,7,9], p=2, q=4:
# node=6: both 2<6 and 4<6 → go left
# node=2: 2==node.val → split (p==node) → return node(2)

def test_harness(fn):
    # (tree_vals, p_val, q_val, expected_lca_val)
    tests = [
        ([6, 2, 8, 0, 4, 7, 9, None, None, 3, 5], 2, 8, 6),
        ([6, 2, 8, 0, 4, 7, 9, None, None, 3, 5], 2, 4, 2),
        ([2, 1], 2, 1, 2),
        ([3, 1, 5], 1, 5, 3),
        ([3, 1, 5], 1, 3, 3),
    ]
    passed = 0
    for *inputs, expected in tests:
        vals, p_val, q_val = inputs
        root = make_tree(vals)
        # find actual node objects by searching the tree
        p_node = bst_search(root, p_val)
        q_node = bst_search(root, q_val)
        got_node = fn(root, p_node, q_node)
        got = got_node.val if got_node else None
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | p={p_val} q={q_val} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(lowest_common_ancestor)
print("lowest_common_ancestor defined.")

<a id='8'></a>
## 8. 🧩 Pattern 4: Delete Node in BST — LC 450

---

```
PROBLEM:
  Given the root of a BST and a key, delete the node with that key.
  Return the root of the modified tree.

TRICK:
  Three cases after finding the target node:
  Case 1: Node is a leaf → return None (detach it)
  Case 2: Node has one child → return that child (promote it)
  Case 3: Node has TWO children → replace value with in-order successor
                                   (leftmost node in right subtree),
                                   then delete the successor from right subtree.

SLOW MOTION TRACE — delete 3 from [5, 3, 6, 2, 4]:

  Tree before:         Tree after:
       5                    5
      / \                  / \
     3   6                4   6
    / \                  /
   2   4                2

  find 3: has two children (2 and 4)
  in-order successor of 3 = leftmost in right subtree(4) = 4
  replace 3's value with 4
  delete 4 from right subtree (case 1: leaf)

SLOW MOTION TRACE — delete 6 from result above:
  find 6: has no children → case 1 → return None → detach

KEY INSIGHT:
  For two-child deletion, swap with in-order SUCCESSOR (not predecessor)
  then recursively delete the successor (which has at most one child).

TIME:  O(h) — find + successor = two root-to-leaf paths
SPACE: O(h) — recursion stack
```

In [ ]:
def delete_node(root, key):
    """
    LC 450 — Delete Node in a BST
    Approach: Navigate to the node, then handle 3 structural cases.
    Args:
        root (TreeNode): root of a valid BST.
        key (int): value of the node to delete.
    Returns:
        TreeNode: root of the modified BST (may change if root is deleted).
    Time:  O(h) — find is O(h), successor find is O(h)
    Space: O(h) — recursion stack depth = tree height
    """
    if not root:
        return None          # key not found — nothing to delete

    if key < root.val:
        root.left = delete_node(root.left, key)   # target is in the left drawer
    elif key > root.val:
        root.right = delete_node(root.right, key) # target is in the right drawer
    else:
        # found the node to delete — now handle the 3 cases
        if not root.left:
            return root.right  # case 1/2: no left child → promote right subtree
        if not root.right:
            return root.left   # case 2: no right child → promote left subtree

        # case 3: two children — find in-order successor (leftmost in right subtree)
        successor = root.right
        while successor.left:              # walk left until we hit the floor
            successor = successor.left     # successor = smallest in right subtree
        root.val = successor.val           # steal the successor's value
        root.right = delete_node(root.right, successor.val) # delete the original successor

    return root

# Slow motion — delete 3 from [5,3,6,2,4]:
# delete_node(5, 3): 3<5 → go left
# delete_node(3, 3): found! has two children (2,4)
#   successor = rightmost-left of 3's right = 4 (no left child)
#   root.val = 4; delete_node(right_subtree=4, 4)
#   delete_node(4, 4): found! leaf → return None
# result: node labeled 4, left=2, right=None

def test_harness(fn):
    tests = [
        ([5, 3, 6, 2, 4, None, 7], 3, [5, 4, 6, 2, None, None, 7]),  # delete node with 2 children
        ([5, 3, 6, 2, 4, None, 7], 0, [5, 3, 6, 2, 4, None, 7]),     # key not in tree
        ([5, 3, 6, 2, 4, None, 7], 7, [5, 3, 6, 2, 4]),              # delete leaf
        ([5, 3, 6, 2, 4, None, 7], 5, [6, 3, 7, 2, 4]),              # delete root
    ]
    passed = 0
    for *inputs, expected in tests:
        vals, key = inputs
        root = make_tree(vals)
        got_root = fn(root, key)
        got = tree_to_list(got_root)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | tree={vals} key={key} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(delete_node)
print("delete_node defined.")

<a id='9'></a>
## 9. The BST Decision Map

```
QUESTION TYPE                         KEY TECHNIQUE               LC PROBLEMS
─────────────────────────────────────────────────────────────────────────────
Validate BST                          Pass [lo, hi] bounds         98
Kth smallest in BST                   Iterative in-order + count   230
Lowest common ancestor (BST)          Navigate via BST property    235
Delete node in BST                    3-case structural surgery    450
Insert node                           Recurse left/right, return   701
Search in BST                         Follow value comparison      700
BST from sorted array                 Midpoint as root, recurse    108
Range sum in BST                      Prune branches by range      938
Convert BST to sorted doubly linked   In-order thread pointers     426
BST Iterator                          Lazy in-order via stack      173

WHEN BST > GENERIC BINARY TREE:
  LCA:    O(h) vs O(n) — BST property eliminates half the search
  Search: O(h) vs O(n) — no need to explore both children
  KthMin: O(h+k) vs O(n) — stop early, no full traversal needed
```

<a id='10'></a>
## 10. Interview Cheat Sheet

**1. When to reach for BST:**

| Signal | What to Do |
|--------|------------|
| "validate BST" | pass lo/hi bounds through recursion |
| "kth smallest" | iterative in-order, stop at k |
| "lowest common ancestor" | exploit BST property O(h) |
| "delete/insert node" | recurse left or right, return root |
| "in-order" or "sorted output" | in-order traversal is sorted |
| "balanced BST from sorted" | midpoint recursion |

**2. The O(h) operations — memorize these:**

```python
# search
while node:
    if target == node.val: return node
    node = node.left if target < node.val else node.right

# insert
def insert(root, val):
    if not root: return TreeNode(val)
    if val < root.val: root.left = insert(root.left, val)
    else:              root.right = insert(root.right, val)
    return root

# validate (pass bounds)
def valid(node, lo, hi):
    if not node: return True
    if not (lo < node.val < hi): return False
    return valid(node.left, lo, node.val) and valid(node.right, node.val, hi)

# inorder successor (leftmost in right subtree)
succ = node.right
while succ.left: succ = succ.left
```

**3. Common templates:**

```python
# TEMPLATE: ITERATIVE IN-ORDER (early exit possible)
stack, node = [], root
while stack or node:
    while node:                # drill left
        stack.append(node)
        node = node.left
    node = stack.pop()         # visit
    # --- do something with node.val here ---
    node = node.right          # move to right subtree

# TEMPLATE: LCA IN BST
node = root
while node:
    if p.val < node.val and q.val < node.val: node = node.left
    elif p.val > node.val and q.val > node.val: node = node.right
    else: return node

# TEMPLATE: DELETE (3 cases)
def delete(root, key):
    if not root: return None
    if key < root.val:   root.left  = delete(root.left,  key)
    elif key > root.val: root.right = delete(root.right, key)
    else:
        if not root.left:  return root.right
        if not root.right: return root.left
        succ = root.right
        while succ.left: succ = succ.left
        root.val = succ.val
        root.right = delete(root.right, succ.val)
    return root
```

**4. Gotchas to not forget:**

```
❌  Checking only parent-child relationship to validate — must pass range bounds
❌  Forgetting that in-order of BST = SORTED — this is the core property
❌  Using the predecessor instead of successor in delete (both work but pick one)
❌  Forgetting to return root after insert/delete (tree structure rebuilt on return)
✅  For validation, use float('-inf') / float('inf') as initial bounds
✅  For kth smallest, iterative in-order lets you stop early — O(h+k) not O(n)
✅  For LCA, BST gives you O(h) — never write O(n) solution if tree is a BST
✅  After delete with 2 children: steal successor's VALUE, then delete the successor NODE
```

## Summary Map

```
                    🌳 BINARY SEARCH TREE
                            │
              ┌─────────────┼─────────────┐
              │             │             │
         VALIDATE        SEARCH        MODIFY
              │             │         ┌──┴──┐
        Pass [lo,hi]   O(h) walk    INSERT DELETE
        down recursion  left/right   return  3-case
        LC 98           root         root    surgery
                                     LC 701  LC 450
              │
         IN-ORDER
         TRAVERSAL
              │
       ┌──────┴──────┐
       │             │
    KTH SMALLEST   LCA (BST)
    Iterative +    First split
    countdown      point = LCA
    LC 230         LC 235

CORE RULE:
  left subtree < node < right subtree  (ALL descendants, not just children)
  In-order traversal ALWAYS yields sorted output
  BST operations are O(h): O(log n) balanced, O(n) skewed
```

---
*End of Binary Search Tree Master Guide — Sean Edition*